# IE-OPT-10 · 带容量的社区店配送

**场景**：河西冷链从配送中心给四家社区店送箱装货物。杏园店 4 箱、柏溪店 3 箱、桂巷店 5 箱、榆湾店 4 箱。每辆车最多装 10 箱，从中心出发并回到中心。两辆车足够覆盖 16 箱需求。目标是总行驶距离最短。

距离和需求都是合成教学数据。车辆相同，所以「1 号车走甲线、2 号车走乙线」和反过来是同一个方案。

练习用，不计正式成绩。

## 1. 模型要点

- 下标 $k$ 表示车辆。$x_{ijk}$ 为 0-1，表示车辆 $k$ 是否从 $i$ 直接开到 $j$。
- 每家店恰好被某辆车进入一次。每辆车从中心最多出发一次，回来的次数和出发次数相同。
- 在每辆车上，店与店之间的进入次数等于离开次数，避免半截路线。
- $u_{ik}$ 表示车辆 $k$ 到达店 $i$ 时已经装上的箱数（含这家店）。$d_i \le u_{ik} \le Q$。
- 若车辆 $k$ 从店 $i$ 开到店 $j$，则 $u_{jk} \ge u_{ik} + d_j$。箱数沿路线递增，既限制载重，也排除不经过中心的小圈。

模型不包含卸货时间、时间窗和司机工时。

In [ ]:
from itertools import permutations
from pyscipopt import Model, quicksum

stops = ["河西中心", "杏园店", "柏溪店", "桂巷店", "榆湾店"]
demand = [0, 4, 3, 5, 4]
capacity = 10
n_vehicles = 2
distance = [
    [0, 8, 6, 9, 7],
    [8, 0, 5, 11, 4],
    [6, 5, 0, 8, 10],
    [9, 11, 8, 0, 6],
    [7, 4, 10, 6, 0],
]
assert sum(demand) <= capacity * n_vehicles
assert all(distance[i][j] == distance[j][i] for i in range(5) for j in range(5))

## 2. 求解两辆车的路线

In [ ]:
def solve_cvrp(distance, demand, capacity, n_vehicles):
    n = len(distance)
    customers = range(1, n)
    model = Model("IE-OPT-10 cvrp")
    model.hideOutput(True)
    x = {(i, j, k): model.addVar(vtype="B", name=f"arc_{i}_{j}_{k}")
         for k in range(n_vehicles) for i in range(n) for j in range(n) if i != j}
    u = {(i, k): model.addVar(lb=demand[i], ub=capacity, vtype="C", name=f"load_{i}_{k}")
         for i in customers for k in range(n_vehicles)}
    model.setObjective(quicksum(distance[i][j] * x[i, j, k] for i, j, k in x), "minimize")
    for j in customers:
        model.addCons(quicksum(x[i, j, k] for k in range(n_vehicles) for i in range(n) if i != j) == 1)
    for k in range(n_vehicles):
        model.addCons(quicksum(x[0, j, k] for j in customers) <= 1)
        model.addCons(quicksum(x[i, 0, k] for i in customers) == quicksum(x[0, j, k] for j in customers))
        for i in customers:
            model.addCons(quicksum(x[i, j, k] for j in range(n) if j != i)
                          == quicksum(x[j, i, k] for j in range(n) if j != i))
        for i in customers:
            for j in customers:
                if i != j:
                    model.addCons(u[j, k] >= u[i, k] + demand[j] - capacity * (1 - x[i, j, k]))
    model.optimize()
    status = str(model.getStatus())
    if status != "optimal":
        raise RuntimeError(f"Solver status is {status}; optimality not certified")
    routes = []
    for k in range(n_vehicles):
        nxt = {}
        for i in range(n):
            for j in range(n):
                if i != j and model.getVal(x[i, j, k]) > 0.5:
                    nxt[i] = j
        if 0 not in nxt:
            continue
        route = [0]
        for _ in range(n):
            nxt_stop = nxt.get(route[-1])
            if nxt_stop is None:
                break
            route.append(nxt_stop)
            if nxt_stop == 0:
                break
        routes.append(route)
    return {"objective": float(model.getObjVal()), "routes": routes}

result = solve_cvrp(distance, demand, capacity, n_vehicles)
print("最短总距离:", result["objective"])
for route in result["routes"]:
    load = sum(demand[i] for i in route if i != 0)
    print(" → ".join(stops[i] for i in route), "装载", load)

## 3. 按客户排列再切开

四家店的全排列只有 24 种。把一个排列从左到右切成若干段，每段的需求不超过载重，段内顺序就是这辆车的路线。车辆相同，段与段的先后不影响总距离。

In [ ]:
def oracle_min_distance(distance, demand, capacity):
    customers = list(range(1, len(distance)))
    best = [float("inf")]

    def route_length(order):
        load = sum(demand[i] for i in order)
        if not order or load > capacity:
            return None
        points = [0, *order, 0]
        return sum(distance[points[i]][points[i + 1]] for i in range(len(points) - 1))

    def split(order, start, acc):
        if acc >= best[0]:
            return
        if start == len(order):
            best[0] = acc
            return
        for end in range(start, len(order)):
            length = route_length(order[start:end + 1])
            if length is None:
                break
            split(order, end + 1, acc + length)

    for order in permutations(customers):
        split(order, 0, 0)
    if best[0] == float("inf"):
        raise RuntimeError("no feasible routes")
    return best[0]

oracle = oracle_min_distance(distance, demand, capacity)
print("独立划分的最短距离:", oracle)
assert abs(result["objective"] - oracle) < 1e-6
seen = []
for route in result["routes"]:
    assert route[0] == 0 and route[-1] == 0
    customers = [i for i in route if i != 0]
    assert len(customers) == len(set(customers))
    assert sum(demand[i] for i in customers) <= capacity
    seen.extend(customers)
assert sorted(seen) == [1, 2, 3, 4]
print("通过：求解器目标与路线划分一致，每家店恰好配送一次且未超载")

## 4. 可以接着改什么

1. 把载重改成 9，看原来能拼在一辆车上的两家店是否必须拆开。
2. 给每家店加一个最晚到达时刻，就是带时间窗的车辆路径，本轮没有单独 Notebook。
3. 四家店的枚举只能核对这个例子。店数增加后，要用同一套车辆流模型，而不是继续枚举排列。